In [1]:
%load_ext autoreload
%autoreload 2

## 1. Imports

In [2]:
import os
import sys

sys.path.append("..")

import random

import numpy as np
import torch
from comet_ml import Experiment
from src.costs.lse import MLPLSECost
from src.models.gmm_based import GMMEOT
from src.plotting.distributions import plot_swiss_roll
from src.plotting.parameters import (
    plot_A_parameters,
    plot_B_parameters,
    plot_Z_parameters,
)
from src.samplers.from_dataset import DatasetSampler
from src.samplers.primary import StandardNormalSampler, SwissRollSampler
from src.utils.metrics import (
    compute_BW_UVP,
    compute_metrics,
    compute_mmd,
    compute_sinkhorn_divergence,
)
from src.utils.paired import get_paired_sampler, match_gaussian_and_swiss_roll
from src.utils.train import compute_loss, update_average
from tqdm import tqdm

In [3]:
device = torch.device(f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [4]:
torch.set_default_device(device)
dtype = torch.float64
torch.torch.set_default_dtype(dtype)

## 2. Config

In [5]:
from configs.gmm_based.cost import MLPLSECostConfig
from configs.gmm_based.dataset import DatasetConfig, MiniBatchConfig
from configs.gmm_based.optimizer import OptPairedConfig, OptUnpairedConfig
from configs.gmm_based.train import TrainConfig

In [6]:
# Data
Q_X_UNPAIRED_SAMPLES = 1024
R_Y_UNPAIRED_SAMPLES = 1024
P_XY_PAIRED_SAMPLES = 128

# Optimizer
LR_PAIRED = 3e-4
LR_UNPAIRED = 1e-3

# Sampler
PAIRED_BATCH_SIZE = 128
UNPAIRED_BATCH_SIZE = 128

G_FUNC = lambda x: x
NUM_METRIC_SAMPLES = 1024

# Train
MAX_STEPS = 100001
INIT_BY_SAMPLES = True

# Potential
Y_DIM = 2
N_POTENTIALS = 50

# Cost
M_POTENTIALS = 25
LOG_V_M_HIDDEN_CHANNELS = [M_POTENTIALS]
B_M_HIDDEN_CHANNELS = [M_POTENTIALS * Y_DIM]

In [7]:
dataset_config = DatasetConfig(
    P_XY_paired=P_XY_PAIRED_SAMPLES, Q_X_unpaired=Q_X_UNPAIRED_SAMPLES, R_Y_unpaired=R_Y_UNPAIRED_SAMPLES
)
minibatch_config = MiniBatchConfig()

cost_config = MLPLSECostConfig(
    m_potentials=M_POTENTIALS,
    log_v_m_hidden_channels=LOG_V_M_HIDDEN_CHANNELS,
    b_m_hidden_channels=B_M_HIDDEN_CHANNELS,
)
EXP_META_INFO = (
    f"M_POTENTIALS_{M_POTENTIALS}_"
    + f"LOG_V_M_HIDDEN_CHANNELS_{LOG_V_M_HIDDEN_CHANNELS}_"
    + f"B_M_HIDDEN_CHANNELS_{B_M_HIDDEN_CHANNELS}_"
)

opt_unpaired_config = OptUnpairedConfig(lr=LR_UNPAIRED)
opt_paired_config = OptPairedConfig(lr=LR_PAIRED)

train_config = TrainConfig(
    steps_to=MAX_STEPS, paired_batch_size=PAIRED_BATCH_SIZE, unpaired_batch_size=UNPAIRED_BATCH_SIZE
)

In [8]:
torch.manual_seed(train_config.seed)
np.random.seed(train_config.seed)
random.seed(train_config.seed)

## 3. Create data and samplers

In [9]:
X_sampler = StandardNormalSampler(dim=dataset_config.x_dim, device=device)
Y_sampler = SwissRollSampler(dim=dataset_config.y_dim, device=device)

In [10]:
X_paired_train = X_sampler.sample(P_XY_PAIRED_SAMPLES)
Y_paired_train = match_gaussian_and_swiss_roll(Y_sampler, X_paired_train, 1, g_func=G_FUNC).squeeze(1)
X_paired_test = X_sampler.sample(P_XY_PAIRED_SAMPLES)
Y_paired_test = match_gaussian_and_swiss_roll(Y_sampler, X_paired_test, 1, g_func=G_FUNC).squeeze(1)

100%|█████████████████████████████████████████████████████████████████████| 128/128 [00:00<00:00, 2860.47it/s]


In [11]:
pd_train_sampler = get_paired_sampler(
    X_paired_train, Y_paired_train, train_config.paired_batch_size, dataset_config.P_XY_paired, device
)

In [12]:
X_unpaired_test = X_sampler.sample(dataset_config.P_XY_paired)
Y_unpaired_test = Y_sampler.sample(dataset_config.P_XY_paired)

In [13]:
if dataset_config.Q_X_unpaired > 0:
    source_data = X_sampler.sample(dataset_config.Q_X_unpaired)
    usd_sampler = DatasetSampler(source_data, device=device)  # usd - unpaired source data
else:
    usd_sampler = DatasetSampler(X_paired_train, device=device)

if dataset_config.R_Y_unpaired > 0:
    target_data = Y_sampler.sample(dataset_config.R_Y_unpaired)
    utd_sampler = DatasetSampler(target_data, device=device)  # utd - unpaired target data
else:
    utd_sampler = DatasetSampler(Y_paired_train, device=device)

## 4. Model initialization

In [14]:
cost = MLPLSECost(**cost_config.model_dump())

In [15]:
model = GMMEOT(
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    cost=cost,
).to(dtype)

if INIT_BY_SAMPLES:
    model.init_a_by_samples(Y_sampler.sample(N_POTENTIALS))

In [16]:
# For EMA update
if train_config.ema_update:
    model_copy = GMMEOT(
        y_dim=Y_DIM,
        n_potentials=N_POTENTIALS,
        cost=cost,
    ).to(dtype)

## 5. Optimizers initialization

In [17]:
# unpaired_params_to_update = [model._log_w_n, model._a_n, model._log_A_n]

unpaired_params_to_update = [
    {"params": [model._log_w_n, model._a_n], "lr": opt_unpaired_config.lr},
    {"params": [model._log_A_n], "lr": opt_unpaired_config.lr * 0.1} # Slow down variance collapse
]

D_opt_unpaired = torch.optim.Adam(unpaired_params_to_update, **opt_unpaired_config.model_dump())

In [18]:
D_opt_paired = torch.optim.Adam(model.cost.parameters(), **opt_paired_config.model_dump(), weight_decay=1e-4)

In [19]:
# TODO: refactor this config
EXP_NAME = (
    "GMMEOT_Swiss_Roll_"
    + f"P_XY_PAIRED_{dataset_config.P_XY_paired}_"
    + f"Q_X_UNPAIRED_{dataset_config.Q_X_unpaired}_"
    + f"R_Y_UNPAIRED_{dataset_config.R_Y_unpaired}_"
    + f"LR_PAIRED_{opt_paired_config.lr}_"
    + f"LR_UNPAIRED_{opt_unpaired_config.lr}_"
    + f"M_{M_POTENTIALS}_"
    + f"N_{N_POTENTIALS}_"
    + f"MINIBATCH_COST_{minibatch_config.cost_function}_"
    + EXP_META_INFO
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    X_DIM=dataset_config.x_dim,
    Y_DIM=dataset_config.y_dim,
    M_POTENTIALS=M_POTENTIALS,
    N_POTENTIALS=N_POTENTIALS,
    D_LR_PAIRED=opt_paired_config.lr,
    D_LR_UNPAIRED=opt_unpaired_config.lr,
    BATCH_SIZE=train_config.unpaired_batch_size,
    P_XY_PAIRED_SAMPLES=dataset_config.P_XY_paired,
    Q_X_UNPAIRED_SAMPLES=dataset_config.Q_X_unpaired,
    R_Y_UNPAIRED_SAMPLES=dataset_config.R_Y_unpaired,
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH, exist_ok=True)

In [20]:
if train_config.steps_from > 0:
    D_opt_unpaired.load_state_dict(
        torch.load(os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{train_config.steps_from}.pt"))
    )
    D_opt_paired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_paired_{train_config.steps_from}.pt")))

## 6. Model training

In [21]:
starting_points = torch.tensor([[-2.0, 0.0], [2.0, 2.0], [0.0, 0.0]])
num_ending_points = 64

In [22]:
num_starting_points_paired = 5
indices = random.choices(range(dataset_config.P_XY_paired), k=num_starting_points_paired)
starting_points_paired = X_paired_train[indices]
ending_points_paired = Y_paired_train[indices]

In [23]:
gt_Y_points = match_gaussian_and_swiss_roll(Y_sampler, X_paired_test, 64, g_func=G_FUNC)
gt_Y_points_for_metrics = match_gaussian_and_swiss_roll(Y_sampler, X_paired_test, NUM_METRIC_SAMPLES, g_func=G_FUNC)

100%|█████████████████████████████████████████████████████████████████████| 128/128 [00:00<00:00, 2730.81it/s]


In [24]:
metrics_dict = {
    "mmd": lambda x, y: compute_mmd(x, y),
    "W_2": lambda x, y: compute_sinkhorn_divergence(x, y),
    "BW_UVP": lambda x, y: compute_BW_UVP(x, y),
}

In [ ]:
experiment = Experiment(
    project_name="Light-GCOT-Swiss-Roll",
    auto_output_logging=False,
    parse_args=False,
)
experiment.set_name(EXP_NAME)
experiment.log_parameters(config)

for step in tqdm(range(train_config.steps_from, train_config.steps_to)):
    # training loop
    D_opt_unpaired.zero_grad()

    X = usd_sampler.sample(train_config.unpaired_batch_size)
    Y = utd_sampler.sample(train_config.unpaired_batch_size)

    output_unpaired = model.compute_unpaired_loss(X, Y)
    D_loss_unpaired = output_unpaired["loss"]

    D_opt_paired.zero_grad()
    X_paired, Y_paired = pd_train_sampler.sample(train_config.paired_batch_size)

    output_paired = model.compute_paired_loss(X_paired, Y_paired)
    D_loss_paired = output_paired["loss"]

    D_loss = D_loss_unpaired + D_loss_paired
    D_loss.backward()

    # Clip gradients for both networks
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    torch.nn.utils.clip_grad_norm_(cost.parameters(), max_norm=1.0)

    D_opt_paired.step()
    D_opt_unpaired.step()

    if train_config.ema_update:
        update_average(model_copy, model, 0.99)
        model = model_copy
    else:
        model = model

    if step % train_config.log_every == 0:
        experiment.log_metric("Unpaired loss", D_loss_unpaired.item(), step=step)
        experiment.log_metric("Paired loss", D_loss_paired.item(), step=step)
        experiment.log_metric("Loss", D_loss, step=step)
        experiment.log_metric(
            "Train paired loss",
            compute_loss(model, X_paired_train, Y_paired_train, X_paired_train, Y_paired_train),
            step=step,
        )
        experiment.log_metric(
            "Test paired loss",
            compute_loss(model, X_paired_test, Y_paired_test, X_paired_test, Y_paired_test),
            step=step,
        )
        experiment.log_metric(
            "Test unpaired loss",
            compute_loss(model, X_unpaired_test, Y_unpaired_test, X_paired_test, Y_paired_test),
            step=step,
        )

        experiment.log_metric(r"$-f^c(x)$", -output_unpaired["f_c"].mean().item(), step=step)
        experiment.log_metric(r"$-f(y)$", -output_unpaired["f"].mean().item(), step=step)
        experiment.log_metric(f"lam_min(A_n)", torch.min(output_unpaired["A_n"]), step=step)
        experiment.log_metric(f"lam_max(A_n)", torch.max(output_unpaired["A_n"]), step=step)
        unconditional_metrics, conditional_metrics = compute_metrics(
            models_dict={"GMMEOT": model},
            metrics_dict=metrics_dict,
            X_sampler=X_sampler,
            Y_sampler=Y_sampler,
            starting_points=starting_points,
            gt_Y_points=gt_Y_points_for_metrics,
            num_samples=NUM_METRIC_SAMPLES,
            experiment=experiment,
        )

    if step % train_config.plot_every == 0:
        plot_A_parameters(model, experiment=experiment)
        plot_B_parameters(model.cost, starting_points, experiment=experiment)
        if num_starting_points_paired > 0:
            plot_Z_parameters(
                model, starting_points, starting_points_paired, ending_points_paired, experiment=experiment
            )
        else:
            plot_Z_parameters(model, starting_points, experiment=experiment)
        plot_swiss_roll(
            {f"P={P_XY_PAIRED_SAMPLES}, Q={Q_X_UNPAIRED_SAMPLES}, R={R_Y_UNPAIRED_SAMPLES}": model},
            X_sampler,
            Y_sampler,
            X_paired,
            Y_paired,
            starting_points,
            gt_Y_points,
            experiment=experiment,
        )

        torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{step}.pt"))

torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{MAX_STEPS}.pt"))
torch.save(D_opt_paired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_paired_{MAX_STEPS}.pt"))
torch.save(D_opt_unpaired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{MAX_STEPS}.pt"))

experiment.end()

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/muxaujl11110/light-gcot-swiss-roll/62f8f979da004b6d953d280e90588709

  0%|                                                                              | 0/100001 [00:00<?, ?it/s]/trinity/home/m.persiyanov/miniconda3/envs/light-gcot/lib/python3.12/site-packages/torch/utils/_device.py:78: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return func(*args, **kwargs)
  0%|                                                                   | 44/100001 [00:07<1:38:29, 16.92it/s]/trinity/home/m.persiyanov/mi

Code for parameter search.

In [ ]:
import scrapbook as sb

# Choose the metric you want Optuna to minimize (e.g., your W_2 or BW_UVP calculation)
# For example, grabbing the final W_2 score:
final_mmd_score = unconditional_metrics["mmd"] # (Adjust this variable name to match your actual final metric)

# Glue it to the notebook so Optuna can read it
sb.glue("target_metric", final_mmd_score)